In [2]:
!pip install -q -U langchain langchain-community langchain-google-genai faiss-cpu pypdf

In [18]:
from google.colab import userdata
import os

os.environ['GOOGLE_API_KEY']=userdata.get("GEMINI_API_KEYS")
MODEL="gemini-3.5-flash-lite"
EMB_MODEL="gemini-embedding-001"

In [6]:
from google.colab import files
uploaded=files.upload()
pdf_name=list(uploaded.keys())[0]
print(f"PDF:{pdf_name} uploaded sucessfully")

Saving Livestock_Wearable_Full_Requirements (2).pdf to Livestock_Wearable_Full_Requirements (2).pdf
PDF:Livestock_Wearable_Full_Requirements (2).pdf uploaded sucessfully


In [15]:
from langchain_community.document_loaders import PyPDFLoader
docs=PyPDFLoader(pdf_name).load()
print("No of PAGES NUMBER: ",len(docs))
print("FEW CONTENT OF PAGE1: ",(docs[0].page_content)[:100])
print("META DATA OF PAGE1: ",docs[0].metadata)

No of PAGES NUMBER:  6
FEW CONTENT OF PAGE1:  Livestock Wearable Monitoring System
Complete Component Requirements & Bill of Materials (incl. Gate
META DATA OF PAGE1:  {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-27T08:15:49+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-09-27T08:15:49+00:00', 'subject': '(unspecified)', 'title': 'Livestock Wearable System - Full Requirements & BOM', 'trapped': '/False', 'source': 'Livestock_Wearable_Full_Requirements (2).pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}


In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(chunk_size=800,chunk_overlap=120)
chunks=splitter.split_documents(docs)
print(f"No of chunks:{len(chunks)}")
print(chunks)

No of chunks:13
[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-27T08:15:49+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-09-27T08:15:49+00:00', 'subject': '(unspecified)', 'title': 'Livestock Wearable System - Full Requirements & BOM', 'trapped': '/False', 'source': 'Livestock_Wearable_Full_Requirements (2).pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='Livestock Wearable Monitoring System\nComplete Component Requirements & Bill of Materials (incl. Gateway / Data Link)\n1. Ear Tag Components\n Sl.\nNo.\nComponent\nRecommended Part/Model\nQty\nPurpose\nKey Specification\nCost\n(■)\n1\nMicrocontroller module\nESP32-C3-MINI-1(U)\n1\nMain controller, BLE communication\nRISC-V, integrated PCB antenna, BLE 5.0,\nlow-power modes\n220\n2\nTemperature sensor\nDS18B20 (waterproof probe)\n1\nBody temperature measurement\n1-Wire digital, ±0.5°C, -10 to 85°C\n80\n3\nPull-up resist

In [21]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
embeddings=GoogleGenerativeAIEmbeddings(model=EMB_MODEL)
vectorstore=FAISS.from_documents(chunks,embeddings)

In [32]:
retriever=vectorstore.as_retriever(search_kwargs={'k':5})

In [69]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import (ChatPromptTemplate,MessagesPlaceholder)
from langchain_core.runnables import (RunnablePassthrough,RunnableWithMessageHistory)
from langchain_core.chat_history import (BaseChatMessageHistory,InMemoryChatMessageHistory)
from langchain_core.output_parsers import StrOutputParser
llm=ChatGoogleGenerativeAI(model=MODEL,max_output_tokens=2024)
prompt=ChatPromptTemplate([
    ('system',"""
        You are a helpful question-answering assistant.

        Answer the question using ONLY the provided context.

        Use the conversation history to understand
        follow-up questions.

        If the answer is not available in the context,
        say:

        "I could not find this information in the document."

        Do not make up information.
        """
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    ('human',"context:{context}---------question:{question}")
])
def format_docs(docs):
  return "\n-------------\n".join(d.page_content for d in docs)

rag_chain=(
    RunnablePassthrough.assign(
        context=(lambda x:x['question']) | retriever | format_docs
    )
    | prompt
    | llm
    | StrOutputParser()
)
store={}
def get_session_history(session_id:str)->BaseChatMessageHistory:
  if session_id not in store:
    store[session_id]=InMemoryChatMessageHistory()
  return store[session_id]
conversation_chain=RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key='question',
    history_messages_key="chat_history"
)

import uuid
session_id=str(uuid.uuid4())
print("Enter bye,exit,quit to terminate session")
def chat(question):
  response=conversation_chain.invoke(
      {"question":question},
      config={
          'configurable':{"session_id":session_id}
      }
  )
  return response
while True:
  question=input("Enter the query: ")
  if question in ['exit','bye','quit']:
    print("EXITED")
    break
  answer=chat(question)
  print(answer)


/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Enter bye,exit,quit to terminate session
Enter the query: tell me cost of this product
Based on the provided document, the cost for a total 1 full system is **4,863**. 

Additionally, the cost per additional animal (Ear Tag + Collar + RFID tag only) is approximately **■2,543**.
Enter the query: explain me project in point
Based on the provided document, here is an explanation of the project in points:

* **System Design & Components:** The project is an animal monitoring system made up of several interconnected components, including an Ear Tag, a Collar, an RFID System, a Gateway (BLE-WiFi Bridge), and a Charging Station.
* **Data Communication Path:**
  * **Ear Tag to Collar:** Uses BLE 5.0 advertisement to transmit temperature readings.
  * **Collar to Gateway:** Uses BLE 4.2 advertisement/GATT to send JSON payloads containing ID, latitude, longitude, temperature, activity, health, and battery status.
  * **RFID Reader to Interface controller:** Uses a UART serial connection to send 